In [1]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import json
from tqdm import tqdm
import torch
import torch.nn as nn
import pickle
import shutil
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
!pip install kaggle
import kagglehub

In [4]:
path = kagglehub.dataset_download("risangbaskoro/wlasl-processed")

print("Path to dataset files:", path)

100%|██████████| 4.82G/4.82G [03:02<00:00, 28.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5


In [5]:
import json

json_file_path = "/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/WLASL_v0.3.json"

with open(json_file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print("Data type:", type(data))
if isinstance(data, list) and len(data) > 0:
    print("First item type:", type(data[0]))
    print("First item keys:", data[0].keys() if isinstance(data[0], dict) else data[0])
elif isinstance(data, dict):
    print("Dictionary keys:", data.keys())

Data type: <class 'list'>
First item type: <class 'dict'>
First item keys: dict_keys(['gloss', 'instances'])


In [6]:
import json
import pandas as pd

json_file_path = "/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/WLASL_v0.3.json"

with open(json_file_path, 'r', encoding='utf-8') as f:
    wlasl_data = json.load(f)

records = []
for entry in wlasl_data:
    gloss = entry.get('gloss')
    instances = entry.get('instances', [])  # Updated to 'instances' (plural)
    for inst in instances:
        records.append({
            'gloss': gloss,
            'video_id': inst.get('video_id'),
            'fps': inst.get('fps'),
            'split': inst.get('split'),
            'url': inst.get('url')
        })

df_wlasl = pd.DataFrame(records)
print(f"Successfully loaded {len(df_wlasl)} video instances across {df_wlasl['gloss'].nunique()} unique glosses.")
display(df_wlasl.head())

Successfully loaded 21083 video instances across 2000 unique glosses.


,gloss,video_id,fps,split,url
0,book,69241,25,train,http://aslbricks.org/New/ASL-Videos/book.mp4
1,book,65225,25,train,https://aslsignbank.haskins.yale.edu/dictionar...
2,book,68011,25,train,https://www.youtube.com/watch?v=0UsjUE-TXns
3,book,68208,25,train,https://www.youtube.com/watch?v=1QOYOZ3g-aY
4,book,68012,25,train,https://www.youtube.com/watch?v=aGtIHKEdCds


In [7]:


dataset_path = "/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5"

video_files = []
video_dirs = []

for root, dirs, files in os.walk(dataset_path):
    for dir_name in dirs:
        if 'video' in dir_name.lower() or 'raw' in dir_name.lower():
            video_dirs.append(os.path.join(root, dir_name))
    for file in files:
        if file.endswith(('.mp4', '.avi', '.mov', '.mkv')):
            video_files.append(os.path.join(root, file))

print(f"Found {len(video_files)} video files directly.")
print(f"Found potential video directories: {video_dirs}")

# If video files are found, show the first few paths
if video_files:
    print("\nSample video paths:")
    for path in video_files[:5]:
        print(path)

Found 11980 video files directly.
Found potential video directories: ['/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/videos']

Sample video paths:
/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/videos/12819.mp4
/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/videos/18036.mp4
/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/videos/44130.mp4
/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/videos/38857.mp4
/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/videos/59829.mp4


In [8]:
videos_dir = "/root/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/videos"

# Map the local file path using the video_id
df_wlasl['local_path'] = df_wlasl['video_id'].astype(str).apply(lambda vid: os.path.join(videos_dir, f"{vid}.mp4"))

# Verify how many files actually exist on disk
df_wlasl['file_exists'] = df_wlasl['local_path'].apply(os.path.exists)

print(f"Total instances in metadata: {len(df_wlasl)}")
print(f"Matching local video files found: {df_wlasl['file_exists'].sum()}")

# Create a clean DataFrame containing only the available local videos
df_filtered = df_wlasl[df_wlasl['file_exists']].reset_index(drop=True)
display(df_filtered.head())

Total instances in metadata: 21083
Matching local video files found: 11980


,gloss,video_id,fps,split,url,local_path,file_exists
0,book,69241,25,train,http://aslbricks.org/New/ASL-Videos/book.mp4,/root/.cache/kagglehub/datasets/risangbaskoro/...,True
1,book,07069,25,train,https://signstock.blob.core.windows.net/signsc...,/root/.cache/kagglehub/datasets/risangbaskoro/...,True
2,book,07068,25,train,https://s3-us-west-1.amazonaws.com/files.start...,/root/.cache/kagglehub/datasets/risangbaskoro/...,True
3,book,07070,25,train,https://media.asldeafined.com/vocabulary/14666...,/root/.cache/kagglehub/datasets/risangbaskoro/...,True
4,book,07099,25,val,http://www.aslsearch.com/signs/videos/book.mp4,/root/.cache/kagglehub/datasets/risangbaskoro/...,True


In [9]:
!pip install mediapipe==0.10.30

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 39.1 MB/s eta 0:00:00
  Attempting uninstall: mediapipe
    Found existing installation: mediapipe 1.0.1
    Uninstalling mediapipe-1.0.1:
      Successfully uninstalled mediapipe-1.0.1


In [10]:
import cv2
import numpy as np
import os
import urllib.request
import mediapipe as mp

# =============================================================================
# 1. Download model files (auto-download if missing)
# =============================================================================
MODEL_DIR = "mediapipe_models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_URLS = {
    "pose": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
    "face": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    "hand": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
}

def download_model(name, url):
    path = os.path.join(MODEL_DIR, os.path.basename(url))
    if not os.path.exists(path):
        print(f"Downloading {name} model...")
        urllib.request.urlretrieve(url, path)
        print(f"Saved: {path}")
    return path

pose_model_path = download_model("pose", MODEL_URLS["pose"])
face_model_path = download_model("face", MODEL_URLS["face"])
hand_model_path = download_model("hand", MODEL_URLS["hand"])

# =============================================================================
# 2. Helper: Create fresh landmarkers (one set per video)
# =============================================================================
BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode

from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions
from mediapipe.tasks.python.vision import FaceLandmarker, FaceLandmarkerOptions
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions

def create_landmarkers():
    """Returns fresh landmarker instances. Call this per video."""
    pose = PoseLandmarker.create_from_options(
        PoseLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=pose_model_path),
            running_mode=VisionRunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5,
        )
    )
    face = FaceLandmarker.create_from_options(
        FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=face_model_path),
            running_mode=VisionRunningMode.VIDEO,
            num_faces=1,
            min_face_detection_confidence=0.5,
            min_face_presence_confidence=0.5,
        )
    )
    hand = HandLandmarker.create_from_options(
        HandLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=hand_model_path),
            running_mode=VisionRunningMode.VIDEO,
            num_hands=2,
            min_hand_detection_confidence=0.5,
            min_hand_presence_confidence=0.5,
        )
    )
    return pose, face, hand

# =============================================================================
# 3. Feature extraction (fresh landmarkers per video, fixed face size)
# =============================================================================
output_features_dir = "extracted_features"
os.makedirs(output_features_dir, exist_ok=True)

def extract_landmarks(video_path):
    # Create fresh instances so timestamps start from 0 for each video
    pose_landmarker, face_landmarker, hand_landmarker = create_landmarkers()

    cap = cv2.VideoCapture(video_path)
    sequence = []
    timestamp_ms = 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_interval_ms = int(1000.0 / fps)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # ---- Pose (33 points * 4 = 132 values) ----
        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
        if pose_result.pose_landmarks:
            landmarks = pose_result.pose_landmarks[0]
            pose = np.array([[lm.x, lm.y, lm.z, lm.visibility]
                             for lm in landmarks]).flatten()
        else:
            pose = np.zeros(33 * 4)

        # ---- Face (468 points * 3 = 1404 values) ----
        # NOTE: New MediaPipe face model returns 478 landmarks.
        # We slice to first 468 to preserve exact backward compatibility.
        face_result = face_landmarker.detect_for_video(mp_image, timestamp_ms)
        if face_result.face_landmarks:
            landmarks = face_result.face_landmarks[0]
            face = np.array([[lm.x, lm.y, lm.z]
                             for lm in landmarks]).flatten()
            # Truncate/pad to exactly 468 landmarks
            if len(face) > 468 * 3:
                face = face[:468 * 3]
            elif len(face) < 468 * 3:
                face = np.pad(face, (0, 468 * 3 - len(face)))
        else:
            face = np.zeros(468 * 3)

        # ---- Hands (21 points * 3 = 63 values each) ----
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)
        lh = np.zeros(21 * 3)
        rh = np.zeros(21 * 3)

        if hand_result.hand_landmarks:
            for idx, landmarks in enumerate(hand_result.hand_landmarks):
                handedness = hand_result.handedness[idx][0].category_name
                arr = np.array([[lm.x, lm.y, lm.z] for lm in landmarks]).flatten()
                if handedness == "Left":
                    lh = arr
                elif handedness == "Right":
                    rh = arr

        # Concatenate: 132 + 1404 + 63 + 63 = 1662
        frame_features = np.concatenate([pose, face, lh, rh])
        sequence.append(frame_features)
        timestamp_ms += frame_interval_ms

    cap.release()
    # Clean up landmarker resources for this video
    pose_landmarker.close()
    face_landmarker.close()
    hand_landmarker.close()

    return np.array(sequence)

# =============================================================================
# 4. Run on your dataset
# =============================================================================
print("Starting test feature extraction...")
for idx, row in df_filtered.head(10).iterrows():
    video_id = row['video_id']
    video_path = row['local_path']
    save_path = os.path.join(output_features_dir, f"{video_id}.npy")

    if os.path.exists(save_path):
        continue

    features = extract_landmarks(video_path)

    if len(features) > 0:
        np.save(save_path, features)
        print(f"Extracted and saved: {video_id}.npy (Shape: {features.shape})")
    else:
        print(f"Warning: No frames processed for video {video_id}")

print("\nTest batch extraction complete!")

Saved: mediapipe_models/pose_landmarker_lite.task
Saved: mediapipe_models/face_landmarker.task
Saved: mediapipe_models/hand_landmarker.task
Starting test feature extraction...


AttributeError: /usr/local/lib/python3.13/dist-packages/mediapipe/tasks/c/libmediapipe.so: undefined symbol: MpErrorFree

In [ ]:
import random

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
print(df_filtered.columns.tolist())

In [ ]:


import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from collections import Counter

# =============================================================================
# 0. Reproducibility & Device
# =============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# =============================================================================
# 1. Vocabulary Builder
# =============================================================================
class Vocabulary:
    def __init__(self, freq_threshold=1):
        self.word2idx = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.freq_threshold = freq_threshold
        self.word_counts = Counter()

    def build_vocabulary(self, sentence_list):
        for sent in sentence_list:
            self.word_counts.update(sent.lower().split())

        for word, count in self.word_counts.items():
            if count >= self.freq_threshold:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def numericalize(self, text):
        tokens = text.lower().split()
        return [self.word2idx.get(t, self.word2idx["<unk>"]) for t in tokens]

    def __len__(self):
        return len(self.word2idx)

# =============================================================================
# 2. Dataset & Collate Function
# =============================================================================
class SignLanguageDataset(Dataset):
    def __init__(self, df, features_dir, vocab):
        self.df = df.reset_index(drop=True)
        self.features_dir = features_dir
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_id = str(row["video_id"])
        text = str(row["gloss"])

        feat_path = os.path.join(self.features_dir, f"{video_id}.npy")
        features = np.load(feat_path).astype(np.float32)

        tokens = [self.vocab.word2idx["<sos>"]]
        tokens += self.vocab.numericalize(text)
        tokens.append(self.vocab.word2idx["<eos>"])

        return torch.FloatTensor(features), torch.LongTensor(tokens)

def collate_fn(batch):
    features, targets = zip(*batch)

    feat_lengths = [f.shape[0] for f in features]
    max_feat_len = max(feat_lengths)
    feat_dim = features[0].shape[1]
    padded_feats = torch.zeros(len(features), max_feat_len, feat_dim)
    for i, f in enumerate(features):
        padded_feats[i, :f.shape[0]] = f

    tgt_lengths = [t.shape[0] for t in targets]
    max_tgt_len = max(tgt_lengths)
    padded_tgts = torch.full((len(targets), max_tgt_len), fill_value=0)
    for i, t in enumerate(targets):
        padded_tgts[i, :t.shape[0]] = t

    return (padded_feats,
            padded_tgts,
            torch.LongTensor(feat_lengths),
            torch.LongTensor(tgt_lengths))

# =============================================================================
# 3. Model: Encoder
# =============================================================================
class Encoder(nn.Module):
    def __init__(self, input_dim=1662, hid_dim=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.input_proj = nn.Linear(input_dim, hid_dim)
        self.lstm = nn.LSTM(hid_dim, hid_dim, n_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

        self.fc_h = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_c = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, x, lengths):
        x = self.dropout(self.input_proj(x))

        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, (hidden, cell) = self.lstm(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)

        hidden = hidden.view(self.n_layers, 2, hidden.size(1), hidden.size(2))
        hidden = torch.cat((hidden[:, 0], hidden[:, 1]), dim=2)
        hidden = self.fc_h(hidden)

        cell = cell.view(self.n_layers, 2, cell.size(1), cell.size(2))
        cell = torch.cat((cell[:, 0], cell[:, 1]), dim=2)
        cell = self.fc_c(cell)

        return outputs, hidden, cell

# =============================================================================
# 4. Model: Attention
# =============================================================================
class Attention(nn.Module):
    def __init__(self, enc_hid_dim=512, dec_hid_dim=512):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        batch_size, src_len, _ = encoder_outputs.shape

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        weights = torch.softmax(attention, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)

        return context, weights

# =============================================================================
# 5. Model: Decoder
# =============================================================================
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim=256, enc_hid_dim=512,
                 dec_hid_dim=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.attention = Attention(enc_hid_dim, dec_hid_dim)

        self.lstm = nn.LSTM(emb_dim + (enc_hid_dim * 2), dec_hid_dim,
                            n_layers, batch_first=True, dropout=dropout)
        self.fc_out = nn.Linear(dec_hid_dim + (enc_hid_dim * 2) + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))

        context, attn_w = self.attention(hidden[-1], encoder_outputs)
        context = context.unsqueeze(1)

        lstm_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        embedded = embedded.squeeze(1)
        output = output.squeeze(1)
        context = context.squeeze(1)

        prediction = self.fc_out(torch.cat((output, context, embedded), dim=1))

        return prediction, hidden, cell, attn_w

# =============================================================================
# 6. Model: Seq2Seq Wrapper
# =============================================================================
class SignLanguageTranslator(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, src_lengths, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src, src_lengths)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell, _ = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

# =============================================================================
# 7. Training & Evaluation
# =============================================================================
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0

    for batch in iterator:
        src, trg, src_len, trg_len = [x.to(device) for x in batch]

        optimizer.zero_grad()
        output = model(src, src_len, trg, teacher_forcing_ratio=0.5)

        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for batch in iterator:
            src, trg, src_len, trg_len = [x.to(device) for x in batch]

            output = model(src, src_len, trg, teacher_forcing_ratio=0.0)

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

# =============================================================================
# 8. Inference: Video -> Text
# =============================================================================
def translate(model, features, vocab, device, max_len=50):
    model.eval()

    if isinstance(features, np.ndarray):
        features = torch.FloatTensor(features)

    features = features.unsqueeze(0).to(device)
    src_len = torch.LongTensor([features.shape[1]]).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(features, src_len)

    inputs = torch.LongTensor([vocab.word2idx["<sos>"]]).to(device)
    outputs = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell, _ = model.decoder(inputs, hidden, cell, encoder_outputs)

        pred = output.argmax(1).item()

        if pred == vocab.word2idx["<eos>"]:
            break

        outputs.append(vocab.idx2word[pred])
        inputs = torch.LongTensor([pred]).to(device)

    return " ".join(outputs)

# =============================================================================
# 9. Main Execution
# =============================================================================
if __name__ == "__main__":
    FEATURES_DIR = "extracted_features"
    BATCH_SIZE = 16
    HID_DIM = 512
    ENC_LAYERS = 2
    DEC_LAYERS = 2
    ENC_DROPOUT = 0.3
    DEC_DROPOUT = 0.3
    N_EPOCHS = 50
    CLIP = 1
    LEARNING_RATE = 0.001

    # -------------------------------------------------------------------------
    # FIX: Filter df_filtered to only videos with extracted .npy features
    # -------------------------------------------------------------------------
    df_filtered["has_features"] = df_filtered["video_id"].apply(
        lambda vid: os.path.exists(os.path.join(FEATURES_DIR, f"{vid}.npy"))
    )
    df_ready = df_filtered[df_filtered["has_features"]].copy()
    df_ready = df_ready.drop(columns=["has_features"])

    print(f"Total videos: {len(df_filtered)}")
    print(f"Videos with features: {len(df_ready)}")

    if len(df_ready) == 0:
        raise ValueError(f"No .npy files found in '{FEATURES_DIR}'. Run feature extraction first!")

    # --- Build Vocabulary ---
    all_sentences = df_ready["gloss"].astype(str).tolist()
    vocab = Vocabulary(freq_threshold=1)
    vocab.build_vocabulary(all_sentences)
    print(f"Vocabulary size: {len(vocab)}")

    # --- Train/Val Split ---
    train_df = df_ready.sample(frac=0.9, random_state=SEED)
    val_df = df_ready.drop(train_df.index)

    train_dataset = SignLanguageDataset(train_df, FEATURES_DIR, vocab)
    val_dataset = SignLanguageDataset(val_df, FEATURES_DIR, vocab)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, collate_fn=collate_fn)

    # --- Initialize Model ---
    enc = Encoder(input_dim=1662, hid_dim=HID_DIM, n_layers=ENC_LAYERS, dropout=ENC_DROPOUT)
    dec = Decoder(output_dim=len(vocab), emb_dim=256, enc_hid_dim=HID_DIM,
                  dec_hid_dim=HID_DIM, n_layers=DEC_LAYERS, dropout=DEC_DROPOUT)

    model = SignLanguageTranslator(enc, dec, device).to(device)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    best_val_loss = float("inf")

    # --- Training Loop ---
    for epoch in range(N_EPOCHS):
        train_loss = train(model, train_loader, optimizer, criterion, CLIP)
        val_loss = evaluate(model, val_loader, criterion)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_sign_language_model.pt")
            print(f"  -> Saved new best model")

        print(f"Epoch {epoch+1:02d}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    # --- Example Inference ---
    print("\n--- Sample Translation ---")
    sample_video_id = val_df.iloc[0]["video_id"]
    sample_text = val_df.iloc[0]["gloss"]
    sample_features = np.load(os.path.join(FEATURES_DIR, f"{sample_video_id}.npy"))

    model.load_state_dict(torch.load("best_sign_language_model.pt"))
    translation = translate(model, sample_features, vocab, device)

    print(f"Video ID : {sample_video_id}")
    print(f"Target   : {sample_text}")
    print(f"Predicted: {translation}")